# SwingSight Club-Head Detector Training

This notebook trains the YOLO model used to locate the club head before SwingSight classifies it. It exports `models/club_detector.pt`, which the app reads automatically.

Use images with one **club-head** bounding box per image. The box should cover the head/face/sole, not the full shaft or golfer.


In [ ]:
import sys
import subprocess

packages = ["ultralytics>=8.0.0", "pyyaml>=6.0.0", "pillow>=10.0.0", "matplotlib>=3.8.0"]
subprocess.check_call([sys.executable, "-m", "pip", "install", *packages, "--no-warn-script-location"])
print("Training packages are ready.")


In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil
import yaml

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the SwingSight-AI project.")

PROJECT_ROOT = find_project_root()
DATASET_DIR = PROJECT_ROOT / "data" / "club_detector"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "experiments" / "club_detector"
MODEL_PATH = PROJECT_ROOT / "models" / "club_detector.pt"

print("Project:", PROJECT_ROOT)
print("Detector dataset:", DATASET_DIR)
print("App model output:", MODEL_PATH)


## Dataset layout

Create this structure before training:

```text
data/club_detector/
  data.yaml
  images/
    train/
    val/
  labels/
    train/
    val/
```

Each image has a matching YOLO text label. Every row is:

`0 center_x center_y width height`

All coordinates are normalized from 0 to 1. There is one class only: `club_head`.


In [ ]:
expected_yaml = {
    "path": str(DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "club_head"},
}
yaml_path = DATASET_DIR / "data.yaml"
if not yaml_path.exists():
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    yaml_path.write_text(yaml.safe_dump(expected_yaml, sort_keys=False), encoding="utf-8")
    print("Created", yaml_path, "- add images and labels, then rerun.")
else:
    print("Using", yaml_path)
    print(yaml.safe_load(yaml_path.read_text(encoding="utf-8")))


In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def split_inventory(split: str) -> dict:
    image_dir = DATASET_DIR / "images" / split
    label_dir = DATASET_DIR / "labels" / split
    images = [path for path in image_dir.rglob("*") if path.suffix.lower() in IMAGE_SUFFIXES] if image_dir.exists() else []
    labels = [path for path in label_dir.rglob("*.txt")] if label_dir.exists() else []
    missing_labels = [path.name for path in images if not (label_dir / (path.stem + ".txt")).exists()]
    return {"images": len(images), "labels": len(labels), "missing_labels": missing_labels[:10]}

inventory = {split: split_inventory(split) for split in ("train", "val")}
inventory


In [ ]:
for split, counts in inventory.items():
    if counts["images"] == 0:
        raise ValueError(f"No {split} images found. Add labeled club-head images first.")
    if counts["missing_labels"]:
        raise ValueError(f"{split} has images without labels: {counts['missing_labels']}")
print("Dataset layout is ready for training.")


## Train and validate

Start with a small YOLO checkpoint and fine-tune it on the club-head class. The run writes metrics and validation images under `outputs/experiments/club_detector`.


In [ ]:
from ultralytics import YOLO

BASE_MODEL = "yolo11n.pt"
EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = -1  # Let Ultralytics choose a safe batch size.

model = YOLO(BASE_MODEL)
train_results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    project=str(OUTPUT_DIR.parent),
    name=OUTPUT_DIR.name,
    exist_ok=True,
    patience=20,
    seed=42,
    pretrained=True,
)
print("Training finished.")


In [ ]:
best_weights = OUTPUT_DIR / "weights" / "best.pt"
if not best_weights.exists():
    raise FileNotFoundError(f"Expected best weights at {best_weights}")

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_weights, MODEL_PATH)
print("Copied trained detector to:", MODEL_PATH)

detector = YOLO(str(MODEL_PATH))
metrics = detector.val(data=str(yaml_path), imgsz=IMAGE_SIZE)
print(metrics)


In [ ]:
# Optional quick visual check on a validation image.
from IPython.display import display
from PIL import Image

val_images = [path for path in (DATASET_DIR / "images" / "val").rglob("*") if path.suffix.lower() in IMAGE_SUFFIXES]
sample = val_images[0]
prediction = detector.predict(source=str(sample), conf=0.25, save=True, project=str(OUTPUT_DIR), name="sample_predictions", exist_ok=True)
display(Image.open(sample))
print("Prediction artifact folder:", OUTPUT_DIR / "sample_predictions")


## App check

After the model has been copied, restart SwingSight and submit a clear club image. The club-detection message should no longer say **club-head detector unavailable**. The exact iron/wedge marking still requires the separate marking model notebook.
